# JAX Transformers Notebook: From Attention Basics to Tiny GPT

This notebook is specialized for transformer-related JAX code with practical details and exercises.


## Prerequisites
- JAX basics (`jit`, `vmap`, PRNG, pytrees)
- Basic matrix multiplication and softmax


In [ ]:
import jax
import jax.numpy as jnp
from jax import random, jit
from flax import linen as nn
import optax

print('JAX:', jax.__version__)


## Section 1: Scaled Dot-Product Attention (Single Head)


In [ ]:
def scaled_dot_product_attention(q, k, v, mask=None):
    # q,k,v: (B,T,D)
    d = q.shape[-1]
    scores = jnp.einsum('btd,bsd->bts', q, k) / jnp.sqrt(d)
    if mask is not None:
        scores = jnp.where(mask > 0, scores, -1e9)
    weights = jax.nn.softmax(scores, axis=-1)
    out = jnp.einsum('bts,bsd->btd', weights, v)
    return out, weights

B,T,D = 2,5,8
key = random.PRNGKey(0)
key,k1,k2,k3 = random.split(key,4)
q = random.normal(k1,(B,T,D))
k = random.normal(k2,(B,T,D))
v = random.normal(k3,(B,T,D))
out,w = scaled_dot_product_attention(q,k,v)
print(out.shape, w.shape)


### Exercise 1
Create a causal mask `(1, T, T)` and verify future positions are masked.


In [ ]:
def causal_mask(T):
    return jnp.tril(jnp.ones((1, T, T), dtype=jnp.float32))

m = causal_mask(T)
_, w_c = scaled_dot_product_attention(q, k, v, mask=m)
print('mask shape:', m.shape)
print('weights[0,0]:', w_c[0,0])


## Section 2: Multi-Head Attention in Practice

Theory says run attention in parallel heads. In practice we:
1. Project once to QKV
2. Reshape into `(B, H, T, Dh)`
3. Compute attention per head
4. Merge heads and output projection


In [ ]:
class MultiHeadSelfAttention(nn.Module):
    d_model: int
    n_heads: int
    dropout_rate: float = 0.0

    @nn.compact
    def __call__(self, x, attn_mask=None, deterministic=True):
        B, T, D = x.shape
        H = self.n_heads
        Dh = D // H
        assert D % H == 0

        qkv = nn.Dense(3 * D, use_bias=False)(x)
        q, k, v = jnp.split(qkv, 3, axis=-1)

        def reshape_heads(t):
            return t.reshape(B, T, H, Dh).transpose(0, 2, 1, 3)  # (B,H,T,Dh)
        q, k, v = map(reshape_heads, (q, k, v))

        scores = jnp.einsum('bhtd,bhsd->bhts', q, k) / jnp.sqrt(Dh)
        if attn_mask is not None:
            # attn_mask expected shape broadcastable to (B,H,T,T)
            scores = jnp.where(attn_mask > 0, scores, -1e9)
        weights = jax.nn.softmax(scores, axis=-1)
        weights = nn.Dropout(self.dropout_rate)(weights, deterministic=deterministic)
        y = jnp.einsum('bhts,bhsd->bhtd', weights, v)  # (B,H,T,Dh)
        y = y.transpose(0, 2, 1, 3).reshape(B, T, D)
        y = nn.Dense(D, use_bias=False)(y)
        return y


In [ ]:
B,T,D,H = 2,8,32,4
key, k = random.split(key)
x = random.normal(k,(B,T,D))
mask = causal_mask(T)[:,None,:,:]  # (1,1,T,T)

mha = MultiHeadSelfAttention(d_model=D, n_heads=H)
params = mha.init(key, x, mask, True)
y = mha.apply(params, x, mask, True)
print('output:', y.shape)


### Exercise 2
Add support for separate padding mask and causal mask and combine them inside MHA.


## Section 3: Transformer Block (Pre-LN)


In [ ]:
class MLP(nn.Module):
    d_model: int
    mlp_ratio: int = 4
    dropout_rate: float = 0.0

    @nn.compact
    def __call__(self, x, deterministic=True):
        hidden = self.d_model * self.mlp_ratio
        x = nn.Dense(hidden)(x)
        x = nn.gelu(x)
        x = nn.Dropout(self.dropout_rate)(x, deterministic=deterministic)
        x = nn.Dense(self.d_model)(x)
        return x

class TransformerBlock(nn.Module):
    d_model: int
    n_heads: int
    dropout_rate: float = 0.0

    @nn.compact
    def __call__(self, x, mask=None, deterministic=True):
        h = nn.LayerNorm()(x)
        h = MultiHeadSelfAttention(self.d_model, self.n_heads, self.dropout_rate)(h, mask, deterministic)
        x = x + nn.Dropout(self.dropout_rate)(h, deterministic=deterministic)

        h2 = nn.LayerNorm()(x)
        h2 = MLP(self.d_model, dropout_rate=self.dropout_rate)(h2, deterministic)
        x = x + nn.Dropout(self.dropout_rate)(h2, deterministic=deterministic)
        return x


## Section 4: Encoder-Only Transformer (classification)


In [ ]:
class EncoderOnlyClassifier(nn.Module):
    vocab_size: int
    d_model: int
    n_heads: int
    n_layers: int
    n_classes: int
    max_len: int

    @nn.compact
    def __call__(self, tokens, deterministic=True):
        B, T = tokens.shape
        x = nn.Embed(self.vocab_size, self.d_model)(tokens)
        pos = self.param('pos_emb', nn.initializers.normal(0.02), (1, self.max_len, self.d_model))
        x = x + pos[:, :T, :]

        for _ in range(self.n_layers):
            x = TransformerBlock(self.d_model, self.n_heads, 0.1)(x, None, deterministic)

        x = nn.LayerNorm()(x)
        cls = jnp.mean(x, axis=1)
        logits = nn.Dense(self.n_classes)(cls)
        return logits


In [ ]:
B,T = 4,16
vocab_size = 100
key, k = random.split(key)
toks = random.randint(k, (B,T), 0, vocab_size)
labels = random.randint(k, (B,), 0, 3)

enc = EncoderOnlyClassifier(vocab_size=vocab_size, d_model=64, n_heads=4, n_layers=2, n_classes=3, max_len=64)
params_enc = enc.init(key, toks, True)
logits = enc.apply(params_enc, toks, True)
print('encoder logits:', logits.shape)


## Section 5: Autoregressive Decoder (causal next-token prediction)


In [ ]:
class TinyGPT(nn.Module):
    vocab_size: int
    d_model: int
    n_heads: int
    n_layers: int
    max_len: int

    @nn.compact
    def __call__(self, tokens, deterministic=True):
        B, T = tokens.shape
        x = nn.Embed(self.vocab_size, self.d_model)(tokens)
        pos = self.param('pos_emb', nn.initializers.normal(0.02), (1, self.max_len, self.d_model))
        x = x + pos[:, :T, :]

        cm = jnp.tril(jnp.ones((1, 1, T, T), dtype=jnp.float32))
        for _ in range(self.n_layers):
            x = TransformerBlock(self.d_model, self.n_heads, 0.1)(x, cm, deterministic)

        x = nn.LayerNorm()(x)
        logits = nn.Dense(self.vocab_size)(x)
        return logits


In [ ]:
def lm_loss(logits, targets, pad_id=0):
    # logits: (B,T,V), targets: (B,T)
    logp = jax.nn.log_softmax(logits, axis=-1)
    nll = -jnp.take_along_axis(logp, targets[..., None], axis=-1).squeeze(-1)
    mask = (targets != pad_id).astype(jnp.float32)
    return (nll * mask).sum() / jnp.maximum(mask.sum(), 1.0)

B,T,V = 4,12,128
key, k1, k2 = random.split(key, 3)
inp = random.randint(k1, (B,T), 1, V)
tgt = random.randint(k2, (B,T), 1, V)

gpt = TinyGPT(vocab_size=V, d_model=64, n_heads=4, n_layers=2, max_len=64)
params_gpt = gpt.init(key, inp, True)
lg = gpt.apply(params_gpt, inp, True)
print('gpt logits:', lg.shape, 'loss:', float(lm_loss(lg, tgt)))


### Exercise 3
Implement greedy decoding for `TinyGPT`:
1. Start from prompt tokens `(B, T0)`
2. Repeatedly predict next token from last position
3. Append until max_new_tokens


In [ ]:
def greedy_decode(params, model, prompt, max_new_tokens=10):
    tokens = prompt
    for _ in range(max_new_tokens):
        logits = model.apply(params, tokens, True)
        next_id = jnp.argmax(logits[:, -1, :], axis=-1, keepdims=True)
        tokens = jnp.concatenate([tokens, next_id], axis=1)
    return tokens

prompt = jnp.array([[1,2,3],[4,5,6]], dtype=jnp.int32)
out = greedy_decode(params_gpt, gpt, prompt, max_new_tokens=5)
print(out.shape)


## Section 6: Training Step Template (LM)


In [ ]:
opt = optax.adamw(1e-3)
opt_state = opt.init(params_gpt)

@jit
def train_step(params, opt_state, x, y):
    def loss_fn(p):
        logits = gpt.apply(p, x, True)
        return lm_loss(logits, y)
    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = opt.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

for step in range(5):
    params_gpt, opt_state, loss = train_step(params_gpt, opt_state, inp, tgt)
    print(f'step={step} loss={float(loss):.4f}')


## Section 7: Common Practical Mistakes

1. Passing Python callables as non-static args into `@jit` functions.
2. Shape changes between steps causing repeated recompilation.
3. Wrong attention mask rank/broadcast shape.
4. Forgetting to shift targets for next-token prediction.
5. Installing/upgrading packages in notebook without kernel restart.
